# Thesis — full corpus (~300 sentences) with augmentation + CRF, journal protocol

1. **Runtime:** Google Colab with **GPU**.
2. **Repo:** public — no GitHub token.
3. **Design:** six encoders × four Exp07 split variants × {`exp07`, `exp07+aug`} × **20** persisted training seeds, on two tracks:
   - **non-CRF:** Exp01 + Exp04, `05_ready`, confidence fusion `06_ready`, six OOF routers `06_*_oof`
   - **CRF:** Exp10 regular BERT-CRF + Exp10 cascade-CRF, confidence fusion `10_fusion_ready`, six OOF routers `10_*_oof`
4. **Protocol:** `--journal-paper` — Exp04 λ grid plus a separate Exp10 CRF-cascade λ grid (resume-aware), then the comparison. Every flag the journal profile would otherwise default (experiments, condition sources, seeds, error-analysis scope) is passed explicitly in cell 7.
5. **OOF routers:** nested 5-fold outer / 4-fold inner CV. On `exp07+aug` conditions each training fold is augmented inside the fold (mask-fill vocabulary from that fold's training sentences only); evaluation folds are never augmented, and the folds are identical to the matching `exp07` condition.
6. **Data:** `data/ner_dataset.pkl` (full labeled corpus) ships with the GitHub clone.
7. **Outputs:** everything — including the exp07 / exp07+aug split JSONs (`--isolate-splits`) — under **`full_aug_crf_run/`** on Drive, so a disconnect loses nothing.
8. **Paper table:** cell 9 builds `unified_protocol_table.xlsx`: fusion vs. base, exp07+aug vs. exp07, and CRF vs. non-CRF, all paired by training seed × outer fold.

**Compute:** ≈48,000 OOF base-model training pairs + 1,920 holdout pairs (≈12× the 150-sentence pilot). The runner finishes one encoder before starting the next (order of `MODELS`). Resume with the **same** lists every time — cell 6 is the single source of truth.

Docs: `thesis_overview.md` Part I §1, Part III, Part IV; `COLAB_README.md`.

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Paths (edit `WORK_ROOT` if needed)

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
# Code + tracked datasets from git on Colab disk (git on Drive often breaks).
REPO_DIR = Path("/content/thesis")
REPO_URL = "https://github.com/menachemsperka/thesis.git"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("WORK_ROOT (run outputs on Drive):", WORK_ROOT)
print("REPO_DIR (clone from GitHub):", REPO_DIR)

## 3 — Clone or update repository (public, no token)

Clones to **`/content/thesis`** (git checkouts on Drive often break). The full corpus `data/ner_dataset.pkl` is tracked in the repo.

In [ ]:
import os
import shutil
import subprocess


def _git_head_ok(repo):
    if not (repo / ".git").is_dir():
        return False
    return subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True).returncode == 0


if _git_head_ok(REPO_DIR):
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    if REPO_DIR.is_dir():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
    print("Cloned repository.")

corpus_pkl = REPO_DIR / "data" / "ner_dataset.pkl"
if not corpus_pkl.is_file():
    raise FileNotFoundError(f"Missing {corpus_pkl}. Ensure origin/main includes data/ner_dataset.pkl.")
print("Corpus from git:", corpus_pkl, "(", corpus_pkl.stat().st_size, "bytes)")

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 4 — Install dependencies

In [ ]:
import os
import subprocess
import sys

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "seqeval", "--no-build-isolation"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies OK.")

## 5 — Colab environment (set before running experiments)

In [ ]:
import os

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
os.environ.setdefault("THESIS_DELETE_MODELS_AFTER_TRAIN", "1")

print("THESIS_RUN_ENV =", os.environ["THESIS_RUN_ENV"])

## 6 — Run configuration (single source of truth — keep identical on every resume)

The checkpoint resume keeps only rows that match the current models / experiments / conditions, so changing these lists between sessions hides (and later re-runs) work. Base-model caches make such re-runs cheap, but the lists should not change mid-study.

In [ ]:
from pathlib import Path

# Isolated run root: checkpoints, seeds, lambda grids, split JSONs (--isolate-splits),
# OOF fold caches and every per-run workbook live here.
OUTPUT_DIR = WORK_ROOT / "full_aug_crf_run"
NUM_SEEDS = 20
# The runner trains one encoder completely before the next, so this order is the priority order.
MODELS = "dictabert,berel,hero,alephbertgimmel,xlm_roberta,mt5"
CONDITION_SOURCES = "exp07,exp07+aug"
NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset.pkl"
EXPERIMENTS = ",".join([
    # Non-CRF track: base pair, B/I repair, confidence fusion, six OOF routers.
    "01", "04", "05_ready", "06_ready",
    "06_svm_oof", "06_svm_kernel_oof", "06_nb_oof", "06_lr_oof", "06_rf_oof", "06_mlp_oof",
    # CRF track: BERT-CRF + cascade-CRF, confidence fusion, six OOF routers.
    "10_regular", "10_cascade", "10_fusion_ready",
    "10_svm_oof", "10_svm_kernel_oof", "10_nb_oof", "10_lr_oof", "10_rf_oof", "10_mlp_oof",
])
# "split" writes one CRF and one non-CRF workbook; "all" needs far more RAM at this scale.
CONSOLIDATED_ERROR_ANALYSIS = "split"

# Keep False on every resume: True draws a brand-new seed list and orphans completed runs.
REGENERATE_TRAINING_SEEDS = False
# True skips both loss-weight grids (Exp04 and Exp10 CRF cascade) and trains with the defaults 10/5.
SKIP_LOSS_CALIBRATION = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_SEEDS:", NUM_SEEDS)
print("MODELS:", MODELS)
print("CONDITION_SOURCES:", CONDITION_SOURCES)
print("EXPERIMENTS:", EXPERIMENTS)
print("NER_DATASET_PKL:", NER_DATASET_PKL)

## 6a — Extend training seeds (idempotent)

On the first run this writes `NUM_SEEDS` random seeds to `training_seeds.json` (the runner would do the same). To grow the study later (e.g. 20 → 30), raise `NUM_SEEDS` in cell 6 and re-run this cell: it **appends** new random seeds and keeps the existing ones, so every completed run stays valid and a plain `--resume` trains only the new seeds. Never use `--regenerate-training-seeds` or `--rerun-experiments` for this.

In [ ]:
import json
import random
from datetime import datetime
from pathlib import Path

seeds_path = Path(OUTPUT_DIR) / "training_seeds.json"
data = json.loads(seeds_path.read_text(encoding="utf-8")) if seeds_path.is_file() else {}
existing = [int(s) for s in data.get("seeds", [])]

if len(existing) >= NUM_SEEDS:
    print(f"training_seeds.json already has {len(existing)} seeds (>= {NUM_SEEDS}); nothing to add.")
else:
    rng = random.SystemRandom()
    seen = set(existing)
    new_seeds = []
    while len(new_seeds) < NUM_SEEDS - len(existing):
        candidate = rng.randint(1, 2_147_483_646)
        if candidate not in seen:
            seen.add(candidate)
            new_seeds.append(candidate)
    data["seeds"] = existing + new_seeds
    data["num_seeds"] = len(data["seeds"])
    data.setdefault(
        "note",
        "Stable training seeds for THESIS_SPLIT_SEED / paired runs. "
        "Reuse this file across reruns; do not edit unless intentional.",
    )
    data["extended_at"] = datetime.now().isoformat(timespec="seconds")
    seeds_path.parent.mkdir(parents=True, exist_ok=True)
    seeds_path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{seeds_path}: {len(existing)} -> {len(data['seeds'])} seeds (added {len(new_seeds)}).")

## 6b — Preflight (GPU, corpus, seed file, Drive space)

If cell 7 fails with a non-zero exit code, scroll up in its output (or open `cell7_run.log` under `OUTPUT_DIR`) for the real `Traceback` / `[FATAL]` line.

In [ ]:
import json
import shutil
import torch
from pathlib import Path

if not NER_DATASET_PKL.is_file():
    raise FileNotFoundError(f"Missing corpus: {NER_DATASET_PKL} — run cell 3 again.")

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARN: CPU runtime is far too slow for this design; switch to Runtime → GPU.")

seeds_path = Path(OUTPUT_DIR) / "training_seeds.json"
if seeds_path.is_file():
    stored = json.loads(seeds_path.read_text(encoding="utf-8")).get("seeds", [])
    print(f"training_seeds.json: {len(stored)} seeds on disk, NUM_SEEDS={NUM_SEEDS}")
    if len(stored) < NUM_SEEDS and not REGENERATE_TRAINING_SEEDS:
        raise ValueError(f"{seeds_path} has {len(stored)} seeds but NUM_SEEDS={NUM_SEEDS}; run cell 6a.")
else:
    print("training_seeds.json: will be created on first run.")

free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
print(f"Free space reported for OUTPUT_DIR: {free_gb:.0f} GB")
if free_gb < 60:
    print("WARN: the OOF fold cache and per-run workbooks reach tens of GB at full scale.")

## 7 — Run cross-comparison (argv one row per argument)

`--journal-paper` runs both loss-weight grids (resume-aware) and implies `--resume`. Explicitly passed `--experiments`, `--condition-sources`, `--num-seeds` and `--consolidated-error-analysis` are kept as given. `--isolate-splits` keeps the exp07 and exp07+aug split JSONs under `OUTPUT_DIR`, so cell 9 can rebuild the folds after a disconnect.

Expect a very long run; on reconnect re-run cells 1–6b, then this cell only.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

env = os.environ.copy()
env["THESIS_RUN_ENV"] = "colab"
env["WANDB_DISABLED"] = "true"
env["THESIS_NER_CSV"] = str(NER_DATASET_PKL)
env["PYTHONUNBUFFERED"] = "1"

argv = [
    sys.executable,
    "run_cross_data_model_comparison.py",
    "--journal-paper",
    "--experiments",
    EXPERIMENTS,
    "--condition-sources",
    CONDITION_SOURCES,
    "--num-seeds",
    str(NUM_SEEDS),
    "--models",
    MODELS,
    "--consolidated-error-analysis",
    CONSOLIDATED_ERROR_ANALYSIS,
    "--output-dir",
    str(OUTPUT_DIR),
    "--isolate-splits",
    "--exp07-source",
    "auto",
    "--base-mode",
    "auto",
    "--resume",
]
if REGENERATE_TRAINING_SEEDS:
    argv.append("--regenerate-training-seeds")
if SKIP_LOSS_CALIBRATION:
    argv.append("--skip-exp04-loss-calibration")

print("Command:")
for i, part in enumerate(argv):
    print(f"  [{i}] {part}")

# Stream line by line (renders live in Colab) and mirror to Drive so a dropped connection keeps the log.
log_path = Path(OUTPUT_DIR) / "cell7_run.log"
with open(log_path, "a", encoding="utf-8") as logf, subprocess.Popen(
    argv,
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

print("exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError(f"Runner exited with code {proc.returncode}. See the [FATAL] / Traceback lines above or {log_path}.")

## 8 — Optional: rebuild Excel/JSON only (disabled for Run All)

Cell 7 already exports everything at the end of a successful run. Use this only to re-export from the checkpoint without training; pass the same lists as cell 7.

In [ ]:
# import subprocess, sys
#
# subprocess.run(
#     [
#         sys.executable, "run_cross_data_model_comparison.py",
#         "--rebuild-from-checkpoint",
#         "--experiments", EXPERIMENTS,
#         "--condition-sources", CONDITION_SOURCES,
#         "--num-seeds", str(NUM_SEEDS),
#         "--models", MODELS,
#         "--consolidated-error-analysis", CONSOLIDATED_ERROR_ANALYSIS,
#         "--output-dir", str(OUTPUT_DIR),
#         "--isolate-splits",
#     ],
#     cwd=str(REPO_DIR),
#     check=True,
# )
print("Cell 8 (rebuild-only) skipped intentionally — see the note above.")

## 9 — Unified single-protocol results table (the file to write the paper from)

Re-scores **every** method of both tracks on the **same** pooled outer-test tokens and writes `unified_protocol_table.xlsx` under `OUTPUT_DIR`. No retraining.

| Sheet | Question |
|---|---|
| `fold_summary` | Main-table cells: mean ± SD F1 over (seed × fold) units, per model / track / condition / method |
| `fusion_vs_base`, `paired_fold_tests` | Does fusion beat its own track's base models? (Holm per model × track × condition) |
| `paired_condition_tests` | Does in-fold augmentation help? exp07+aug − exp07 on identical folds (Holm per model × track × variant) |
| `paired_track_tests` | Does CRF help? CRF − non-CRF per method family on identical folds (Holm per model × condition) |
| `token_error_summary` | FP / FN / type / boundary counts per method |

`--splits-dir` points at the isolated `exp07/splits` (fold membership is condition-specific; exp07+aug conditions map to their original exp07 variant). Reading ~11,500 OOF workbooks over the Drive mount is slow, so they are copied to local disk first.

**Check `n_pairs`:** 20 seeds × 5 outer folds = **100**. `fold_size_mismatches` must be **0** — otherwise the two sides of a contrast did not share fold partitions.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir(REPO_DIR)

SPLITS_DIR = Path(OUTPUT_DIR) / "exp07" / "splits"
RUNNER_XLSX = Path(OUTPUT_DIR) / "cross_comparison_latest.xlsx"
UNIFIED_XLSX = Path(OUTPUT_DIR) / "unified_protocol_table.xlsx"
COPY_OOF_WORKBOOKS_LOCALLY = True

if not (SPLITS_DIR / "split_meta.json").is_file():
    raise FileNotFoundError(f"Missing {SPLITS_DIR}; cell 7 writes it on its first run (--isolate-splits).")

oof_dir = Path(OUTPUT_DIR)
if COPY_OOF_WORKBOOKS_LOCALLY:
    oof_dir = Path("/content/oof_workbooks")
    for src in sorted((Path(OUTPUT_DIR) / "experiment_outputs").glob("*_oof")):
        shutil.copytree(src, oof_dir / src.name, dirs_exist_ok=True)
        print("copied", src.name)

argv_unified = [
    sys.executable,
    "build_unified_protocol_table.py",
    "--oof-dir",
    str(oof_dir),
    "--splits-dir",
    str(SPLITS_DIR),
    "--output",
    str(UNIFIED_XLSX),
]
# Carry the runner's paper sheets (lambda grids, loss config, holdout table) into the same workbook.
if RUNNER_XLSX.is_file():
    argv_unified += ["--reference-xlsx", str(RUNNER_XLSX)]

print("Command:")
for i, part in enumerate(argv_unified):
    print(f"  [{i}] {part}")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
log_path = Path(OUTPUT_DIR) / "cell9_unified.log"
with open(log_path, "a", encoding="utf-8") as logf, subprocess.Popen(
    argv_unified,
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"build_unified_protocol_table.py exited with code {proc.returncode}. See {log_path}.")
print("\nWrote:", UNIFIED_XLSX)

## 9a — Inspect the unified workbook

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

sheets = pd.read_excel(Path(OUTPUT_DIR) / "unified_protocol_table.xlsx", sheet_name=None)
print("Sheets:", {name: df.shape for name, df in sheets.items()})


def show(name, columns, sort_by, max_rows=60):
    frame = sheets.get(name)
    if frame is None or frame.empty:
        print(f"\n{name}: empty")
        return None
    print(f"\n=== {name} ({len(frame)} rows; first {max_rows}) ===")
    view = frame[[c for c in columns if c in frame.columns]].sort_values(sort_by)
    print(view.head(max_rows).to_string(index=False))
    return frame


show("fold_summary",
     ["model", "track", "base_split_condition", "method", "f1_mean_pm_sd", "n_seeds", "n_folds"],
     ["model", "track", "base_split_condition", "method"])
fusion_tests = show("paired_fold_tests",
     ["model", "track", "base_split_condition", "contrast", "n_pairs", "mean_delta_f1",
      "wilcoxon_p_holm", "ttest_rel_p_holm", "sig_05_holm"],
     ["model", "track", "base_split_condition", "contrast"])
aug_tests = show("paired_condition_tests",
     ["model", "track", "variant", "method", "n_pairs", "fold_size_mismatches", "aug_mean_f1",
      "plain_mean_f1", "mean_delta_f1", "wilcoxon_p_holm", "sig_05_holm"],
     ["model", "track", "variant", "method"])
crf_tests = show("paired_track_tests",
     ["model", "base_split_condition", "method_family", "n_pairs", "fold_size_mismatches",
      "crf_mean_f1", "non_crf_mean_f1", "mean_delta_f1", "wilcoxon_p_holm", "sig_05_holm"],
     ["model", "base_split_condition", "method_family"])
show("token_error_summary",
     ["model", "track", "base_split_condition", "method", "n_seeds", "false_positive",
      "false_negative", "type_error", "boundary_error", "token_error_rate"],
     ["model", "track", "base_split_condition", "method"])

expected_pairs = NUM_SEEDS * 5
for name, frame in (("paired_fold_tests", fusion_tests), ("paired_condition_tests", aug_tests), ("paired_track_tests", crf_tests)):
    if frame is not None:
        print(f"{name}: n_pairs observed {sorted(frame['n_pairs'].unique().tolist())} (expected {expected_pairs})")
for name, frame in (("paired_condition_tests", aug_tests), ("paired_track_tests", crf_tests)):
    if frame is not None and int(frame["fold_size_mismatches"].sum()):
        print(f"WARNING: {name} has fold-size mismatches; those contrasts are not paired on identical folds.")

## 10 — List key output files

In [ ]:
from pathlib import Path

out = Path(OUTPUT_DIR)
checks = [
    "run_manifest.json",
    "training_seeds.json",
    "exp07/splits/split_meta.json",
    "exp07_augmented/splits_train_vocab/split_meta.json",
    "exp04_lambda_grid_cache/exp04_lambda_selection.json",
    "exp10_lambda_grid_cache/exp10_lambda_selection.json",
    "cross_comparison_progress_latest.json",
    "cross_comparison_latest.xlsx",
    "unified_protocol_table.xlsx",
    "consolidated_error_analysis_non_exp10_latest.xlsx",
    "consolidated_error_analysis_exp10_latest.xlsx",
]
for rel in checks:
    print(f"{rel}: {'OK' if (out / rel).is_file() else 'missing'}")